# Comparaison des références TD1

Exécuter les deux cellules pour comparer les requêtes SQLite, le calcul indépendant sur CSV et les montants enregistrés. Les montants sont en centimes HT.

In [1]:
"""Build SQL references and independently check CSV totals; no model API calls."""
import csv
import json
import re
import sqlite3
from datetime import date, datetime
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'reference-td1.csv').exists():
    raise FileNotFoundError('Ouvrir le notebook depuis la racine du projet.')


def read_csv(name):
    with (ROOT / 'data' / name).open(encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def csv_total(question, orders, customers, prices):
    # Parse the original wording independently of the SQL specifications below.
    timestamps = re.findall(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}', question)
    period = [datetime.fromisoformat(t) for t in timestamps]
    reference = re.search(r'reference date (\d{4}-\d{2}-\d{2})', question)
    products = re.search(r'products? (.*?) only', question)
    allowed = set(re.findall(r'\b[a-e]\b', products.group(1))) if products else None
    total = 0
    for row in orders:
        placed = datetime.fromisoformat(row['date'])
        if period and not period[0] <= placed < period[1]:
            continue
        if allowed is not None and row['product_id'] not in allowed:
            continue
        if reference or 'at the time of the order' in question:
            on = date.fromisoformat(reference.group(1)) if reference else placed.date()
            birth = customers[row['customer_id']]
            # Birthday interval, independent of SQL's completed-age subtraction.
            if not birth.replace(year=birth.year + 25) <= on < birth.replace(year=birth.year + 26):
                continue
        total += int(row['quantity_product']) * prices[row['product_id']]
    return total


def main():
    questions = json.loads((ROOT / 'questions-td1.json').read_text(encoding='utf-8'))['questions']
    with (ROOT / 'reference-td1.csv').open(encoding='utf-8', newline='') as f:
        references = list(csv.DictReader(f))
    expected_ids = [q['id'] for q in questions]
    if len(references) != 20 or [r['case_id'] for r in references] != expected_ids:
        raise ValueError('Le fichier de reference doit contenir C01 a C20 dans cet ordre.')
    orders = read_csv('orders.csv')
    customers = {r['customer_id']: date.fromisoformat(r['birth_date']) for r in read_csv('customers.csv')}
    prices = {r['product_id']: int(r['price']) for r in read_csv('product.csv')}
    headers = ['Question', 'SQL execute', 'Calcul CSV', 'Reference', 'Ecart SQL/CSV', 'Statut']
    table = []
    def number(value):
        return format(value, ',').replace(',', ' ')
    with sqlite3.connect((ROOT / 'data/boutique.sqlite').as_uri() + '?mode=ro', uri=True) as con:
        con.execute('PRAGMA query_only = 1')
        for question, ref in zip(questions, references):
            cursor = con.execute(ref['sql'])
            result = cursor.fetchall()
            if [d[0] for d in cursor.description] != ['revenue_cents'] or len(result) != 1 or len(result[0]) != 1 or type(result[0][0]) is not int:
                raise ValueError(f"{ref['case_id']}: resultat SQL invalide")
            actual = result[0][0]
            independent = csv_total(question['question'], orders, customers, prices)
            recorded = int(ref['revenue_cents'])
            ok = actual == independent == recorded
            table.append([ref['case_id'], number(actual), number(independent), number(recorded), number(actual - independent), 'OK' if ok else 'ECHEC'])
    widths = [max(len(headers[i]), *(len(row[i]) for row in table)) for i in range(len(headers))]
    separator = '+' + '+'.join('-' * (w + 2) for w in widths) + '+'
    def show(row):
        cells = [value.rjust(widths[i]) if 1 <= i <= 4 else value.ljust(widths[i]) for i, value in enumerate(row)]
        print('| ' + ' | '.join(cells) + ' |')
    print('\nCOMPARAISON DES REFERENCES TD1')
    print('Montants en centimes EUR HT - base SQLite ouverte en lecture seule.\n')
    print(separator)
    show(headers)
    print(separator)
    for row in table:
        show(row)
    print(separator)
    success = sum(row[-1] == 'OK' for row in table)
    print(f'\nBilan : {success}/20 concordances (SQL = CSV = reference).')
    print('SQL : requetes du fichier reference-td1.csv executees sur la base.')
    print('CSV : calcul Python separe sur les trois fichiers sources.')
    return 0 if success == 20 else 1


In [2]:
main()



COMPARAISON DES REFERENCES TD1
Montants en centimes EUR HT - base SQLite ouverte en lecture seule.

+----------+-------------+------------+-----------+---------------+--------+
| Question | SQL execute | Calcul CSV | Reference | Ecart SQL/CSV | Statut |
+----------+-------------+------------+-----------+---------------+--------+
| C01      |     792 800 |    792 800 |   792 800 |             0 | OK     |
| C02      |     590 320 |    590 320 |   590 320 |             0 | OK     |
| C03      |     936 350 |    936 350 |   936 350 |             0 | OK     |
| C04      |   2 826 250 |  2 826 250 | 2 826 250 |             0 | OK     |
| C05      |      38 480 |     38 480 |    38 480 |             0 | OK     |
| C06      |           0 |          0 |         0 |             0 | OK     |
| C07      |      13 970 |     13 970 |    13 970 |             0 | OK     |
| C08      |     605 580 |    605 580 |   605 580 |             0 | OK     |
| C09      |     487 980 |    487 980 |   487 980 | 

## Comparaison des references avec le modele E
Lecture des traces reelles sauvegardees dans `traces_E.CSV`.

In [3]:
import csv
from pathlib import Path
from IPython.display import HTML, display
from html import escape

root = Path.cwd()
with (root / "reference-td1.csv").open(encoding="utf-8", newline="") as f:
    references = {r["case_id"]: int(r["revenue_cents"]) for r in csv.DictReader(f)}
with (root / "traces_E.CSV").open(encoding="utf-8", newline="") as f:
    attempts = list(csv.DictReader(f))
# Retenir la derniere tentative reelle du modele E pour chaque question.
latest = {}
for row in attempts:
    if row["nature"] == "real" and row["provider"] == "openrouter" and row["model_requested"] == "dots-studio/dots-3-note-preview:free":
        latest[row["case_id"]] = row

def amount(value):
    return format(value, ",").replace(",", " ")

labels = {"correct": "Correct", "incorrect": "Incorrect", "sql_invalid": "SQL invalide", "sql_rejected": "SQL refuse"}
body = []
correct = 0
for case, reference in references.items():
    row = latest.get(case)
    value = int(row["value"]) if row and row["value"] != "" else None
    valid = row is not None and row["verdict"] in ("correct", "incorrect")
    matches = valid and value == reference
    correct += matches
    verdict = ("correct" if matches else "incorrect") if valid else row["verdict"] if row else "absent"
    color = "#dcfce7" if matches else "#fee2e2" if value is not None else "#fef3c7"
    values = [case, amount(reference), amount(value) if value is not None else "--", amount(value - reference) if value is not None else "--", labels.get(verdict, verdict), row["attempts"] if row else "--"]
    cells = "".join('<td style="padding:8px 16px;text-align:' + ('right' if i in (1, 2, 3, 5) else 'left') + '">' + escape(str(v)) + '</td>' for i, v in enumerate(values))
    body.append('<tr style="background:' + color + ';color:#172033;border-bottom:1px solid #ddd">' + cells + '</tr>')
headers = ["Question", "Reference HT (ct)", "Modele E HT (ct)", "Ecart modele - ref.", "Verdict", "Tentative"]
head = ''.join('<th style="padding:10px 16px;text-align:left">' + escape(h) + '</th>' for h in headers)
html = '<h3>References / Modele E - Dots</h3><p><b>' + str(correct) + '/' + str(len(references)) + ' correctes (' + format(100 * correct / len(references), '.0f') + '%)</b> - montants en centimes EUR HT</p>'
html += '<table style="border-collapse:collapse;font-family:system-ui"><thead style="background:#1e293b;color:white"><tr>' + head + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table>'
html += '<p>Vert : correct ; rouge : incorrect ; jaune : aucune valeur exploitable. -- signifie une valeur absente, pas zero. Aucun nouvel appel au modele.</p>'
display(HTML(html))


Question,Reference HT (ct),Modele E HT (ct),Ecart modele - ref.,Verdict,Tentative
C01,792 800,792 800,0,Correct,1
C02,590 320,590 320,0,Correct,1
C03,936 350,936 350,0,Correct,1
C04,2 826 250,2 826 250,0,Correct,1
C05,38 480,38 480,0,Correct,1
C06,0,0,0,Correct,1
C07,13 970,13 970,0,Correct,1
C08,605 580,0,-605 580,Incorrect,1
C09,487 980,47 900,-440 080,Incorrect,1
C10,558 980,0,-558 980,Incorrect,1
